In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression


In [2]:
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


os.chdir("/notebooks/Thesis-DataLogging")
print("Changed working directory to:", os.getcwd())

Current working directory: /notebooks/Thesis-DataLogging
Files in this directory: ['CSV_SQLite.ipynb', '.ipynb_checkpoints', 'FlowLog_20251127_113337.csv', 'flow_data.db', '40min_print_SI383820251201085118_PartStatistics_all.csv', 'Airbearing_print_271125_SI383820251128093848_PartStatistics_all.csv', 'Merged_Data_log_OT.csv', 'Timestamp_LA_INDEX_27.csv', 'Timestamp_LA_INDEX.csv', 'Merged_Data_log_OT_27.csv', '27_Merged_Data_log_OT.csv', 'FlowLog_20251203_123354.csv', 'FlowLog_20251203_145757.csv', '80um-reference-job-SI383820251203121902_PartStatistics_all.csv', 'merged_3_12_25.csv', 'Timestamp_LA_INDEX_03.csv', '03_Merged_Data_log_OT.csv', 'SI383820251203121902 (2).pdf', '03_12_25_Merged_Data_log_OT.csv', 'extracted_data.csv', 'Fullmerged.csv', 'FlowLog_20251215_095220.csv', 'DoE.ipynb', 'Run_2.pdf', 'Run_1.pdf', 'Sensor_Calibration(Sheet1).csv', 'ML.ipynb', 'plc_operational_data_with_corrected_flows.csv', 'Backend.ipynb']
Changed working directory to: /notebooks/Thesis-DataLogging


In [3]:
# Current working directory should be .../notebooks
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR

INPUT_PLC_CSV = DATA_DIR / "FlowLog_20251215_095220.csv"
INPUT_CALIB_CSV = DATA_DIR / "Sensor_Calibration(Sheet1).csv"
OUTPUT_CSV = DATA_DIR / "plc_operational_data_with_corrected_flows.csv"

# Safety checks
assert INPUT_PLC_CSV.exists(), f"Missing file: {INPUT_PLC_CSV}"
assert INPUT_CALIB_CSV.exists(), f"Missing file: {INPUT_CALIB_CSV}"

print("Paths resolved correctly")


Paths resolved correctly


In [4]:
def load_operational_data(path):
    df = pd.read_csv(path)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df = df.sort_values("Timestamp").reset_index(drop=True)
    return df


def load_calibration_data(path):
    """
    Columns expected:
    SensorType, RAW_value, Flow_Lmin
    """
    return pd.read_csv(path)


plc_df = load_operational_data(INPUT_PLC_CSV)
calib_df = load_calibration_data(INPUT_CALIB_CSV)

print("Operational data shape:", plc_df.shape)
print("Calibration data shape:", calib_df.shape)


Operational data shape: (165587, 9)
Calibration data shape: (23, 3)


In [5]:
"""
def build_sensor_training_data(
    op_df,
    cal_df,
    sensor_name,
    raw_col,
    flow_col
):
    # Operational data
    df_op = op_df[[raw_col, flow_col]].copy()
    df_op["source"] = "operational"

    # Calibration anchors
    df_cal = cal_df[cal_df["SensorType"] == sensor_name].copy()
    df_cal = df_cal.rename(
        columns={"RAW_value": raw_col, "Flow_Lmin": flow_col}
    )
    df_cal["source"] = "calibration"

    return pd.concat([df_op, df_cal], ignore_index=True)
"""

'\ndef build_sensor_training_data(\n    op_df,\n    cal_df,\n    sensor_name,\n    raw_col,\n    flow_col\n):\n    # Operational data\n    df_op = op_df[[raw_col, flow_col]].copy()\n    df_op["source"] = "operational"\n\n    # Calibration anchors\n    df_cal = cal_df[cal_df["SensorType"] == sensor_name].copy()\n    df_cal = df_cal.rename(\n        columns={"RAW_value": raw_col, "Flow_Lmin": flow_col}\n    )\n    df_cal["source"] = "calibration"\n\n    return pd.concat([df_op, df_cal], ignore_index=True)\n'

In [6]:
"""
def train_hybrid_sensor_model(df, raw_col, flow_col):
    X = df[[raw_col, flow_col]]
    y = df[flow_col]

    # Give calibration points higher importance
    sample_weight = np.where(
        df["source"] == "calibration", 5.0, 1.0
    )

    model = LGBMRegressor(
        n_estimators=800,
        learning_rate=0.04,
        max_depth=5,
        monotone_constraints=[1, 1],
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42
    )

    tscv = TimeSeriesSplit(n_splits=5)
    maes = []

    for tr, te in tscv.split(X):
        model.fit(
            X.iloc[tr],
            y.iloc[tr],
            sample_weight=sample_weight[tr]
        )
        preds = model.predict(X.iloc[te])
        maes.append(mean_absolute_error(y.iloc[te], preds))

    print(f"MAE: {np.mean(maes):.4f} L/min")

    model.fit(X, y, sample_weight=sample_weight)
    return model
"""

'\ndef train_hybrid_sensor_model(df, raw_col, flow_col):\n    X = df[[raw_col, flow_col]]\n    y = df[flow_col]\n\n    # Give calibration points higher importance\n    sample_weight = np.where(\n        df["source"] == "calibration", 5.0, 1.0\n    )\n\n    model = LGBMRegressor(\n        n_estimators=800,\n        learning_rate=0.04,\n        max_depth=5,\n        monotone_constraints=[1, 1],\n        subsample=0.9,\n        colsample_bytree=0.9,\n        random_state=42\n    )\n\n    tscv = TimeSeriesSplit(n_splits=5)\n    maes = []\n\n    for tr, te in tscv.split(X):\n        model.fit(\n            X.iloc[tr],\n            y.iloc[tr],\n            sample_weight=sample_weight[tr]\n        )\n        preds = model.predict(X.iloc[te])\n        maes.append(mean_absolute_error(y.iloc[te], preds))\n\n    print(f"MAE: {np.mean(maes):.4f} L/min")\n\n    model.fit(X, y, sample_weight=sample_weight)\n    return model\n'

In [7]:
def build_sensor_training_data_pure(cal_df, sensor_name, raw_col):
    df_cal = cal_df[cal_df["SensorType"] == sensor_name].copy()
    df_cal = df_cal.rename(
        columns={"RAW_value": raw_col, "Flow_Lmin": "True_Lmin"}
    )
    return df_cal  # only calibration
'''
def train_sensor_model(df, raw_col):
    X = df[[raw_col]]
    y = df["True_Lmin"]
    """
    model = LGBMRegressor(
        n_estimators=10,
        learning_rate=0.05,
        max_depth=1,
        num_leaves=2,
        min_data_in_leaf=1,
        subsample=1.0,
        colsample_bytree=1.0,
        random_state=42
    )
    """
    model=LinearRegression()
    model.fit(X, y)
    return model
'''
from sklearn.model_selection import train_test_split

def train_and_evaluate_with_split(df, raw_col, sensor_name):
    X = df[[raw_col]]
    y = df["True_Lmin"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"\nGeneralization performance for {sensor_name} sensor")
    print(f"RMSE : {rmse:.4f} L/min")
    print(f"MAE  : {mae:.4f} L/min")
    print(f"R²   : {r2:.4f}")

    return model

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_regression_model(df, raw_col, model, sensor_name):
    X = df[[raw_col]]
    y_true = df["True_Lmin"]
    y_pred = model.predict(X)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\nEvaluation for {sensor_name} sensor (calibration data)")
    print(f"RMSE : {rmse:.4f} L/min")
    print(f"MAE  : {mae:.4f} L/min")
    print(f"R²   : {r2:.4f}")

    return rmse, mae, r2

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def evaluate_tolerance_metrics(df, raw_col, model, sensor_name, tolerance=0.05):
    X = df[[raw_col]]
    y_true = df["True_Lmin"]
    y_pred = model.predict(X)

    # Within tolerance → 1, else 0
    y_true_cls = np.ones(len(y_true))  # all samples are "valid reference"
    y_pred_cls = (np.abs(y_pred - y_true) <= tolerance * y_true).astype(int)

    acc = accuracy_score(y_true_cls, y_pred_cls)
    f1 = f1_score(y_true_cls, y_pred_cls)
    precision = precision_score(y_true_cls, y_pred_cls)
    recall = recall_score(y_true_cls, y_pred_cls)

    print(f"\nTolerance-based metrics for {sensor_name} sensor (±{int(tolerance*100)}%)")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    return acc, f1

def apply_all_sensor_models_fixed(op_df, cal_df):
    sensor_specs = [
        ("LowFlow", "LowFlowRAW", "LowFlow_Lmin"),
        ("ArgonFlow", "ArgonFlowRAW", "ArgonFlow_Lmin"),
        ("HighFlow", "HighFlowRAW", "HighFlow_Lmin"),
    ]

    for sensor, raw_col, out_col in sensor_specs:
        print(f"\nTraining model for {sensor} sensor")

        train_df = build_sensor_training_data_pure(cal_df, sensor, raw_col)
        #model = train_sensor_model(train_df, raw_col)
        model= train_and_evaluate_with_split(train_df, raw_col, sensor)

        # --- Evaluate on calibration data ---
        evaluate_regression_model(train_df, raw_col, model, sensor)
        
        evaluate_tolerance_metrics(train_df, raw_col, model, sensor, tolerance=0.05)

        # --- Apply to operational data ---
        op_df[out_col] = model.predict(op_df[[raw_col]])

    return op_df

plc_df = apply_all_sensor_models_fixed(plc_df, calib_df)


Training model for LowFlow sensor

Generalization performance for LowFlow sensor
RMSE : 0.0000 L/min
MAE  : 0.0000 L/min
R²   : nan

Evaluation for LowFlow sensor (calibration data)
RMSE : 0.0000 L/min
MAE  : 0.0000 L/min
R²   : 1.0000

Tolerance-based metrics for LowFlow sensor (±5%)
Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1 Score  : 1.0000

Training model for ArgonFlow sensor

Generalization performance for ArgonFlow sensor
RMSE : 2.0815 L/min
MAE  : 2.0583 L/min
R²   : -16.3308

Evaluation for ArgonFlow sensor (calibration data)
RMSE : 1.5955 L/min
MAE  : 1.4161 L/min
R²   : 0.9954

Tolerance-based metrics for ArgonFlow sensor (±5%)
Accuracy  : 0.5000
Precision : 1.0000
Recall    : 0.5000
F1 Score  : 0.6667

Training model for HighFlow sensor

Generalization performance for HighFlow sensor
RMSE : 4.3282 L/min
MAE  : 3.9525 L/min
R²   : 0.9997

Evaluation for HighFlow sensor (calibration data)
RMSE : 2.9906 L/min
MAE  : 2.3237 L/min
R²   : 0.9999

Tolerance-based m

/opt/software/lib/python3.10/site-packages/sklearn/metrics/_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [8]:
"""
def apply_all_sensor_models(op_df, cal_df):

    sensor_specs = [
        ("LowFlow", "LowFlowRAW", "LowFlow", "LowFlow_Lmin"),
        ("ArgonFlow", "ArgonFlowRAW", "ArgonFlow", "ArgonFlow_Lmin"),
        ("HighFlow", "HighFlowRAW", "HighFlow", "HighFlow_Lmin"),
    ]

    for sensor, raw_col, flow_col, out_col in sensor_specs:
        print(f"\nTraining model for {sensor} sensor")

        train_df = build_sensor_training_data(
            op_df, cal_df, sensor, raw_col, flow_col
        )

        model = train_hybrid_sensor_model(
            train_df, raw_col, flow_col
        )

        # Apply model to operational data
        op_df[out_col] = model.predict(
            op_df[[raw_col, flow_col]]
        )

    return op_df


plc_df = apply_all_sensor_models(plc_df, calib_df)
"""

'\ndef apply_all_sensor_models(op_df, cal_df):\n\n    sensor_specs = [\n        ("LowFlow", "LowFlowRAW", "LowFlow", "LowFlow_Lmin"),\n        ("ArgonFlow", "ArgonFlowRAW", "ArgonFlow", "ArgonFlow_Lmin"),\n        ("HighFlow", "HighFlowRAW", "HighFlow", "HighFlow_Lmin"),\n    ]\n\n    for sensor, raw_col, flow_col, out_col in sensor_specs:\n        print(f"\nTraining model for {sensor} sensor")\n\n        train_df = build_sensor_training_data(\n            op_df, cal_df, sensor, raw_col, flow_col\n        )\n\n        model = train_hybrid_sensor_model(\n            train_df, raw_col, flow_col\n        )\n\n        # Apply model to operational data\n        op_df[out_col] = model.predict(\n            op_df[[raw_col, flow_col]]\n        )\n\n    return op_df\n\n\nplc_df = apply_all_sensor_models(plc_df, calib_df)\n'

In [9]:
plc_df.to_csv(OUTPUT_CSV, index=False)

print("Corrected CSV saved to:")
print(OUTPUT_CSV)


Corrected CSV saved to:
/notebooks/Thesis-DataLogging/plc_operational_data_with_corrected_flows.csv


In [10]:
import pandas as pd
import numpy as np

def compute_energy_and_argon_from_csv(
    csv_path,
    start_ts,
    end_ts
):
    """
    Computes energy and argon consumption directly from a PLC log CSV.

    Parameters
    ----------
    csv_path : str or Path
        Path to FlowLog CSV
    start_ts : str
        Start timestamp (e.g. "2025-12-15 09:00:00")
    end_ts : str
        End timestamp (e.g. "2025-12-15 12:00:00")
    """

    # Load data
    df = pd.read_csv(csv_path)

    # Parse timestamps
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])

    # Sort and filter time window
    df = df.sort_values("Timestamp")
    df_window = df[
        (df["Timestamp"] >= pd.to_datetime(start_ts)) &
        (df["Timestamp"] <= pd.to_datetime(end_ts))
    ].copy()

    if len(df_window) < 2:
        raise ValueError("Not enough samples in selected time window.")

    # -------------------------
    # 1) Energy Consumption
    # -------------------------
    energy_start = df_window["Energy_kWh"].iloc[0]
    energy_end = df_window["Energy_kWh"].iloc[-1]
    energy_consumed_kwh = energy_end - energy_start

    # -------------------------
    # 2) Argon Consumption
    # -------------------------
    # Time difference in minutes
    df_window["dt_min"] = df_window["Timestamp"].diff().dt.total_seconds() / 60.0

    # Trapezoidal integration of flow
    df_window["argon_segment_L"] = (
        (df_window["ArgonFlow_Lmin"] + df_window["ArgonFlow_Lmin"].shift(1)) / 2
    ) * df_window["dt_min"]

    argon_consumed_L = df_window["argon_segment_L"].sum(skipna=True)

    return {
        "start_timestamp": start_ts,
        "end_timestamp": end_ts,
        "energy_consumed_kwh": round(energy_consumed_kwh, 4),
        "argon_consumed_litres": round(argon_consumed_L, 2),
        "num_samples": len(df_window)
    }

In [11]:
CSV_PATH = "plc_operational_data_with_corrected_flows.csv"

start_time = "2025-12-15 09:00:00"
end_time   = "2025-12-16 07:40:16"

results = compute_energy_and_argon_from_csv(
    CSV_PATH,
    start_time,
    end_time
)

print("Consumption Summary")
print("-------------------")
print(f"From: {results['start_timestamp']}")
print(f"To  : {results['end_timestamp']}")
print(f"Energy Consumed : {results['energy_consumed_kwh']} kWh")
print(f"Gas Consumed  : {results['argon_consumed_litres']} litres")
print(f"Samples Used    : {results['num_samples']}")

Consumption Summary
-------------------
From: 2025-12-15 09:00:00
To  : 2025-12-16 07:40:16
Energy Consumed : 42.6 kWh
Gas Consumed  : 5178.87 litres
Samples Used    : 75861


In [12]:
import pandas as pd
import numpy as np

def compute_single_purge_argon(csv_path, start_ts, end_ts):

    df = pd.read_csv(csv_path)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df = df.sort_values("Timestamp")

    # Extract purge window
    df_purge = df[
        (df["Timestamp"] >= pd.to_datetime(start_ts)) &
        (df["Timestamp"] <= pd.to_datetime(end_ts))
    ].copy()

    if len(df_purge) < 2:
        raise ValueError("Not enough samples in purge window.")

    # Time delta in minutes
    df_purge["dt_min"] = df_purge["Timestamp"].diff().dt.total_seconds() / 60.0
    df_purge = df_purge[df_purge["dt_min"] > 0]

    # Clip negative or noisy flows
    df_purge["ArgonFlow_Lmin"] = df_purge["ArgonFlow_Lmin"].clip(lower=0)

    # Trapezoidal integration
    df_purge["argon_segment_L"] = (
        (df_purge["ArgonFlow_Lmin"] + df_purge["ArgonFlow_Lmin"].shift(1)) / 2
    ) * df_purge["dt_min"]

    total_argon = df_purge["argon_segment_L"].sum(skipna=True)

    duration_min = (df_purge["Timestamp"].iloc[-1] - df_purge["Timestamp"].iloc[0]).total_seconds() / 60

    avg_flow = df_purge["ArgonFlow_Lmin"].mean()

    print("\nPurge Argon Consumption Report")
    print("------------------------------")
    print(f"Start Time     : {start_ts}")
    print(f"End Time       : {end_ts}")
    print(f"Duration       : {duration_min:.2f} min")
    print(f"Average Flow   : {avg_flow:.2f} L/min")
    print(f"Argon Consumed : {total_argon:.2f} litres")

    return total_argon

In [13]:
CSV_PATH = "plc_operational_data_with_corrected_flows.csv"

argon_used = compute_single_purge_argon(
    CSV_PATH,
    "2025-12-15 09:53:09",
    "2025-12-15 10:21:50"
)


Purge Argon Consumption Report
------------------------------
Start Time     : 2025-12-15 09:53:09
End Time       : 2025-12-15 10:21:50
Duration       : 28.67 min
Average Flow   : 51.34 L/min
Argon Consumed : 1472.12 litres


In [14]:
CSV_PATH = "plc_operational_data_with_corrected_flows.csv"

start_time = "2025-12-16 11:50:09"
                    
end_time   = "2025-12-16 22:44:29"

results = compute_energy_and_argon_from_csv(
    CSV_PATH,
    start_time,
    end_time
)

print("Consumption Summary")
print("-------------------")
print(f"From: {results['start_timestamp']}")
print(f"To  : {results['end_timestamp']}")
print(f"Energy Consumed : {results['energy_consumed_kwh']} kWh")
print(f"Gas Consumed  : {results['argon_consumed_litres']} litres")
print(f"Samples Used    : {results['num_samples']}")

Consumption Summary
-------------------
From: 2025-12-16 11:50:09
To  : 2025-12-16 22:44:29
Energy Consumed : 21.7 kWh
Gas Consumed  : 2468.7 litres
Samples Used    : 37695


In [15]:
CSV_PATH = "plc_operational_data_with_corrected_flows.csv"

argon_used = compute_single_purge_argon(
    CSV_PATH,
    "2025-12-16 12:04:56",
    "2025-12-16 12:21:51"
)


Purge Argon Consumption Report
------------------------------
Start Time     : 2025-12-16 12:04:56
End Time       : 2025-12-16 12:21:51
Duration       : 16.88 min
Average Flow   : 54.39 L/min
Argon Consumed : 918.26 litres


In [16]:
import pandas as pd
import numpy as np

def integrate_flow(df, flow_column):
    df = df.copy()
    df["dt_min"] = df["Timestamp"].diff().dt.total_seconds() / 60.0
    df = df[df["dt_min"] > 0]

    df[flow_column] = df[flow_column].clip(lower=0)

    df["segment_L"] = (
        (df[flow_column] + df[flow_column].shift(1)) / 2
    ) * df["dt_min"]

    return df["segment_L"].sum(skipna=True)


def compute_build_consumption(csv_path, start_ts, end_ts):

    df = pd.read_csv(csv_path)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df = df.sort_values("Timestamp")

    df_build = df[
        (df["Timestamp"] >= pd.to_datetime(start_ts)) &
        (df["Timestamp"] <= pd.to_datetime(end_ts))
    ].copy()

    if len(df_build) < 2:
        raise ValueError("Not enough samples in build window.")

    # PLC calibrated values
    low_plc = integrate_flow(df_build, "LowFlow_Lmin")
    high_plc = integrate_flow(df_build, "HighFlow_Lmin")

    # RAW values (optional comparison)
    low_raw = integrate_flow(df_build, "LowFlowRAW")
    high_raw = integrate_flow(df_build, "HighFlowRAW")

    total_plc = low_plc + high_plc

    return {
        "low_plc_L": low_plc,
        "high_plc_L": high_plc,
        "total_plc_L": total_plc,
        "low_raw_units": low_raw,
        "high_raw_units": high_raw
    }

In [17]:
CSV_PATH = "plc_operational_data_with_corrected_flows.csv"

build_1 = compute_build_consumption(
    CSV_PATH,
    "2025-12-15 09:00:00",
    "2025-12-16 07:40:16"
)

build_2 = compute_build_consumption(
    CSV_PATH,
    "2025-12-16 11:50:09",
    "2025-12-16 22:44:29"
)

In [18]:
def print_build_comparison(b1, b2):

    print("\n===== BUILD 1 =====")
    print(f"LowFlow  : {b1['low_plc_L']:.2f} L")
    print(f"HighFlow : {b1['high_plc_L']:.2f} L")
    print(f"Total    : {b1['total_plc_L']:.2f} L")

    print("\n===== BUILD 2 =====")
    print(f"LowFlow  : {b2['low_plc_L']:.2f} L")
    print(f"HighFlow : {b2['high_plc_L']:.2f} L")
    print(f"Total    : {b2['total_plc_L']:.2f} L")

    print("\n===== PERCENTAGE BREAKDOWN =====")

    for i, b in enumerate([b1, b2], 1):
        low_pct = (b["low_plc_L"] / b["total_plc_L"]) * 100
        high_pct = (b["high_plc_L"] / b["total_plc_L"]) * 100
        print(f"\nBuild {i}:")
        print(f"LowFlow  : {low_pct:.2f}%")
        print(f"HighFlow : {high_pct:.2f}%")

    print("\n===== WHICH BUILD CONSUMED MORE? =====")

    if b1["total_plc_L"] > b2["total_plc_L"]:
        print("Build 1 consumed more gas.")
    else:
        print("Build 2 consumed more gas.")

print_build_comparison(build_1, build_2)


===== BUILD 1 =====
LowFlow  : 5623.97 L
HighFlow : 193.42 L
Total    : 5817.39 L

===== BUILD 2 =====
LowFlow  : 2813.56 L
HighFlow : 87.16 L
Total    : 2900.72 L

===== PERCENTAGE BREAKDOWN =====

Build 1:
LowFlow  : 96.68%
HighFlow : 3.32%

Build 2:
LowFlow  : 97.00%
HighFlow : 3.00%

===== WHICH BUILD CONSUMED MORE? =====
Build 1 consumed more gas.


In [1]:
import pandas as pd
import numpy as np

def compute_energy_and_argon_purge(
    csv_path,
    start_ts,
    end_ts
):
    """
    Computes energy and argon consumption directly from a PLC log CSV.

    Parameters
    ----------
    csv_path : str or Path
        Path to FlowLog CSV
    start_ts : str
        Start timestamp (e.g. "2025-12-15 09:00:00")
    end_ts : str
        End timestamp (e.g. "2025-12-15 12:00:00")
    """

    # Load data
    df = pd.read_csv(csv_path)

    # Parse timestamps
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])

    # Sort and filter time window
    df = df.sort_values("Timestamp")
    df_window = df[
        (df["Timestamp"] >= pd.to_datetime(start_ts)) &
        (df["Timestamp"] <= pd.to_datetime(end_ts))
    ].copy()

    if len(df_window) < 2:
        raise ValueError("Not enough samples in selected time window.")

    # -------------------------
    # 1) Energy Consumption
    # -------------------------
    energy_start = df_window["Energy_kWh"].iloc[0]
    energy_end = df_window["Energy_kWh"].iloc[-1]
    energy_consumed_kwh = energy_end - energy_start

    # -------------------------
    # 2) Argon Consumption
    # -------------------------
    # Time difference in minutes
    df_window["dt_min"] = df_window["Timestamp"].diff().dt.total_seconds() / 60.0

    # Trapezoidal integration of flow
    df_window["argon_segment_L"] = (
        (df_window["ArgonFlow_Lmin"] + df_window["ArgonFlow_Lmin"].shift(1)) / 2
    ) * df_window["dt_min"]

    argon_consumed_L = df_window["argon_segment_L"].sum(skipna=True)

    return {
        "start_timestamp": start_ts,
        "end_timestamp": end_ts,
        "energy_consumed_kwh": round(energy_consumed_kwh, 4),
        "argon_consumed_litres": round(argon_consumed_L, 2),
        "num_samples": len(df_window)
    }

In [2]:
CSV_PATH = "plc_operational_data_with_corrected_flows.csv"

start_time = "2025-12-15 09:53:09"
                    
end_time   = "2025-12-15 10:21:50"

results = compute_energy_and_argon_purge(
    CSV_PATH,
    start_time,
    end_time
)

print("Purge Summary")
print("-------------------")
print(f"From: {results['start_timestamp']}")
print(f"To  : {results['end_timestamp']}")
print(f"Energy Consumed : {results['energy_consumed_kwh']} kWh")
print(f"Gas Consumed  : {results['argon_consumed_litres']} litres")
print(f"Samples Used    : {results['num_samples']}")

Purge Summary
-------------------
From: 2025-12-15 09:53:09
To  : 2025-12-15 10:21:50
Energy Consumed : 0.9 kWh
Gas Consumed  : 1472.43 litres
Samples Used    : 1662


In [3]:
CSV_PATH = "plc_operational_data_with_corrected_flows.csv"

start_time = "2025-12-16 12:04:56"
                    
end_time   = "2025-12-16 12:21:51"

results = compute_energy_and_argon_purge(
    CSV_PATH,
    start_time,
    end_time
)

print("Consumption Summary")
print("-------------------")
print(f"From: {results['start_timestamp']}")
print(f"To  : {results['end_timestamp']}")
print(f"Energy Consumed : {results['energy_consumed_kwh']} kWh")
print(f"Gas Consumed  : {results['argon_consumed_litres']} litres")
print(f"Samples Used    : {results['num_samples']}")

Consumption Summary
-------------------
From: 2025-12-16 12:04:56
To  : 2025-12-16 12:21:51
Energy Consumed : 0.6 kWh
Gas Consumed  : 920.26 litres
Samples Used    : 981
